In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from modules.prepare_data import cut_not_saturday, cut_not_10am, drop_status_6
from modules.calibration import calibrate_global_scale, calibrate_per_group, calibrate_per_group_scale_bias

In [ ]:
# --- CONFIG ---

train_path = "data/features.parquet"       # предобработанные признаки (авг/сент убраны)
test_path  = "data/test_solo_track.parquet"
masks_path = "data/feature_masks.pkl"

RANDOM_STATE    = 42
RIDGE_ALPHA     = 4.0
FORECAST_POINTS = 8
FUTURE_TARGET_COLS = [f"target_step_{s}" for s in range(1, FORECAST_POINTS + 1)]

# ── Дополнительная фильтрация ────────────────────────────────────────────────
APPLY_CUT_NOT_SATURDAY = False
APPLY_CUT_NOT_10AM     = False
APPLY_DROP_STATUS_6    = False

# ── Обрезка начала датасета ──────────────────────────────────────────────────
# None — не обрезать; строка вида "2025-10-05" — брать только с этой даты
TRAIN_START_DATE = None

In [3]:
# --- LOAD DATA ---

train_df = pd.read_parquet(train_path)   # features.parquet — уже с признаками
test_df  = pd.read_parquet(test_path)    # id, route_id, timestamp

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"])
test_df["timestamp"]  = pd.to_datetime(test_df["timestamp"])

train_df = train_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)
test_df  = test_df.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

print("train shape:", train_df.shape)
print("test  shape:", test_df.shape)

train shape: (1702000, 218)
test  shape: (8000, 3)


In [4]:
# --- GENERATE MULTI-STEP TARGETS ---
# sum_1h — исходный таргет (переименован из target_1h в generate_features.py)

route_group = train_df.groupby("route_id", sort=False)
for step in range(1, FORECAST_POINTS + 1):
    train_df[f"target_step_{step}"] = route_group["sum_1h"].shift(-step)

print("Добавлены таргеты:", FUTURE_TARGET_COLS)

Добавлены таргеты: ['target_step_1', 'target_step_2', 'target_step_3', 'target_step_4', 'target_step_5', 'target_step_6', 'target_step_7', 'target_step_8']


## Фильтрация обучающей выборки

In [5]:
# --- FILTER TRAIN ---

filtered = train_df.copy()
if TRAIN_START_DATE is not None:
    filtered = filtered[filtered["timestamp"] >= pd.Timestamp(TRAIN_START_DATE)]
    print(f"Обрезка до {TRAIN_START_DATE}: осталось {len(filtered):,} строк")
if APPLY_CUT_NOT_SATURDAY:
    filtered = cut_not_saturday(filtered)
if APPLY_CUT_NOT_10AM:
    filtered = cut_not_10am(filtered)
if APPLY_DROP_STATUS_6:
    filtered = drop_status_6(filtered)

print("filtered train shape:", filtered.shape)

filtered train shape: (1702000, 226)


## Построение признаков

In [6]:
# --- ЗАГРУЗКА МАСОК И ПОДГОТОВКА ТРЕНИРОВОЧНОГО ДАТАСЕТА ---

feature_masks = joblib.load(masks_path)

# dropna() без subset: убираем и строки с NaN в таргетах (последние 8 на маршрут),
# и строки с NaN в лаговых признаках (первые ~24 строки на маршрут из features.parquet)
train_feat = filtered.dropna().reset_index(drop=True)

print("train_feat shape:", train_feat.shape)
print("Признаков по шагам:")
for step in range(1, FORECAST_POINTS + 1):
    print(f"  step {step:2d}: {feature_masks[step]['n_selected']:3d} признаков")

train_feat shape: (1502000, 226)
Признаков по шагам:
  step  1: 204 признаков
  step  2: 212 признаков
  step  3: 155 признаков
  step  4: 131 признаков
  step  5: 127 признаков
  step  6: 121 признаков
  step  7: 146 признаков
  step  8: 168 признаков


## Обучение Ridge-регрессии (отдельная модель на каждый шаг)

In [7]:
# --- TRAIN ---

models = {}
for step in range(1, FORECAST_POINTS + 1):
    feat_cols = feature_masks[step]["selected"]
    X = train_feat[feat_cols].values
    y = train_feat[f"target_step_{step}"].values
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_STATE)),
    ])
    pipe.fit(X, y)
    models[step] = pipe

    y_train_pred = pipe.predict(X)
    mae = mean_absolute_error(y, y_train_pred)
    print(f"step {step:2d} | train MAE = {mae:,.0f} | признаков: {len(feat_cols)}")

print("\nВсего обучено моделей:", len(models))

step  1 | train MAE = 71,534 | признаков: 204
step  2 | train MAE = 90,047 | признаков: 212
step  3 | train MAE = 93,119 | признаков: 155
step  4 | train MAE = 95,093 | признаков: 131
step  5 | train MAE = 96,131 | признаков: 127
step  6 | train MAE = 96,619 | признаков: 121
step  7 | train MAE = 96,819 | признаков: 146
step  8 | train MAE = 96,828 | признаков: 168

Всего обучено моделей: 8


## Генерация предсказаний на тесте

In [8]:
# --- PREDICT ON TEST ---
# Признаки для теста: последнее наблюдение каждого маршрута из filtered.
# Используем filtered (не train_feat): dropna убрал бы последние строки,
# а они содержат самое свежее состояние, нужное для прогноза.

last_obs = (
    filtered
    .sort_values(["route_id", "timestamp"])
    .groupby("route_id", sort=False)
    .last()
    .reset_index()
    .drop(columns=FUTURE_TARGET_COLS)   # таргеты там NaN — не нужны
)

test_feat = test_df[["id", "route_id", "timestamp"]].copy()
test_feat["step"] = test_feat.groupby("route_id").cumcount() + 1
test_feat = test_feat.merge(
    last_obs.drop(columns=["timestamp"]),
    on="route_id",
    how="left",
).reset_index(drop=True)

y_pred = np.array([
    models[step].predict(test_feat.loc[[i], feature_masks[step]["selected"]].values)[0]
    for i, step in zip(test_feat.index, test_feat["step"])
])

pred_df = pd.DataFrame({"id": test_feat["id"], "y_pred": y_pred})
print(pred_df.shape)
pred_df.head(10)

(8000, 2)


,id,y_pred
0,3920,235063.506176
1,3921,173689.685990
2,3922,189545.304041
3,3923,186247.640473
4,3924,178105.464618
5,3925,168116.499036
6,3926,168892.149544
7,3927,177651.346163
8,5072,91901.543805
9,5073,71410.301543


## Калибровка прогнозов (WAPE + RelBias)

In [9]:
# calibrate_global_scale и calibrate_per_group импортированы из modules/calibration.py

In [10]:
# --- PREPARE VALIDATION DATA FOR CALIBRATION ---
# Последние 20% обучающей выборки по времени — валидация.

val_cutoff_ts = train_feat["timestamp"].quantile(0.8)
val_mask      = train_feat["timestamp"] >= val_cutoff_ts
val_feat      = train_feat.loc[val_mask]

val_preds_list  = []
val_true_list   = []
val_routes_list = []
val_steps_list  = []

for step in range(1, FORECAST_POINTS + 1):
    feat_cols  = feature_masks[step]["selected"]
    y_val_pred = models[step].predict(val_feat[feat_cols].values)
    y_val_true = val_feat[f"target_step_{step}"].values

    val_preds_list.append(y_val_pred)
    val_true_list.append(y_val_true)
    val_routes_list.append(val_feat["route_id"].values)
    val_steps_list.append(np.full(len(val_feat), step, dtype=int))

val_preds_arr  = np.concatenate(val_preds_list)
val_true_arr   = np.concatenate(val_true_list)
val_routes_arr = np.concatenate(val_routes_list)
val_steps_arr  = np.concatenate(val_steps_list)

mask = ~np.isnan(val_true_arr)
val_preds_arr  = val_preds_arr[mask]
val_true_arr   = val_true_arr[mask]
val_routes_arr = val_routes_arr[mask]
val_steps_arr  = val_steps_arr[mask]

print(f"Валидационных наблюдений: {mask.sum():,}  "
      f"(из {len(mask):,} итого, порог: {val_cutoff_ts})")

Валидационных наблюдений: 2,408,000  (из 2,408,000 итого, порог: 2025-10-26 00:30:00)


In [11]:
# --- CALIBRATE: сравниваем три метода ---

def val_score(val_cal):
    wape  = np.abs(val_cal - val_true_arr).sum() / val_true_arr.sum()
    rbias = abs(val_cal.sum() / val_true_arr.sum() - 1)
    return wape + rbias, wape, rbias

SEP = "=" * 55

# 1. Глобальный множитель (1 параметр)
print(SEP); print("1. Глобальный скаляр"); print(SEP)
preds_global, k = calibrate_global_scale(val_true_arr, val_preds_arr, y_pred)

# 2. Per-group additive bias (8 000 параметров)
print(f"\n{SEP}"); print("2. Per-group bias"); print(SEP)
preds_bias, biases = calibrate_per_group(
    val_true_arr, val_preds_arr, val_routes_arr, val_steps_arr,
    y_pred, test_feat["route_id"].values, test_feat["step"].values,
)

# 3. Per-group scale + bias (16 000 параметров)
print(f"\n{SEP}"); print("3. Per-group scale × bias (совместная)"); print(SEP)
preds_sb, scales, biases_sb = calibrate_per_group_scale_bias(
    val_true_arr, val_preds_arr, val_routes_arr, val_steps_arr,
    y_pred, test_feat["route_id"].values, test_feat["step"].values,
)

# --- Итоговое сравнение ---
bias_vec = np.array([biases.get((r, s), 0.0)
                     for r, s in zip(val_routes_arr, val_steps_arr)])
k_vec = np.array([scales.get((r, s), 1.0)
                  for r, s in zip(val_routes_arr, val_steps_arr)])
b_vec = np.array([biases_sb.get((r, s), 0.0)
                  for r, s in zip(val_routes_arr, val_steps_arr)])

m0 = val_score(val_preds_arr)
m1 = val_score(val_preds_arr * k)
m2 = val_score(val_preds_arr + bias_vec)
m3 = val_score(k_vec * val_preds_arr + b_vec)

print(f"\n{'Метод':<30} {'Итого':>8}  {'WAPE':>7}  {'RBias':>7}")
print("-" * 58)
print(f"{'Без калибровки':<30} {m0[0]:>8.6f}  {m0[1]:>7.4f}  {m0[2]:>7.4f}")
print(f"{'1. Глобальный скаляр':<30} {m1[0]:>8.6f}  {m1[1]:>7.4f}  {m1[2]:>7.4f}")
print(f"{'2. Per-group bias':<30} {m2[0]:>8.6f}  {m2[1]:>7.4f}  {m2[2]:>7.4f}")
print(f"{'3. Per-group scale+bias':<30} {m3[0]:>8.6f}  {m3[1]:>7.4f}  {m3[2]:>7.4f}")

results = {"global": (m1[0], preds_global), "bias": (m2[0], preds_bias), "scale_bias": (m3[0], preds_sb)}
best_name, (best_score, best_preds) = min(results.items(), key=lambda x: x[1][0])
print(f"\nВыбран: {best_name}  (val metric = {best_score:.6f})")

pred_df = pd.DataFrame({"id": test_feat["id"], "y_pred": best_preds})

1. Глобальный скаляр
Глобальный множитель k = 0.992768
Метрика на валидации ДО:    0.335353  (WAPE=0.3281, RBias=0.0073)
Метрика на валидации ПОСЛЕ: 0.327505  (WAPE=0.3275, RBias=0.0000)

2. Per-group bias
Статус: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Метрика на валидации ДО:    0.335353
Метрика на валидации ПОСЛЕ: 0.326045

3. Per-group scale × bias (совместная)
Статус: ABNORMAL: 
Метрика на валидации ДО:    0.335353
Метрика на валидации ПОСЛЕ: 0.325500

Метод                             Итого     WAPE    RBias
----------------------------------------------------------
Без калибровки                 0.335353   0.3281   0.0073
1. Глобальный скаляр           0.327505   0.3275   0.0000
2. Per-group bias              0.326045   0.3260   0.0000
3. Per-group scale+bias        0.325500   0.3255   0.0000

Выбран: scale_bias  (val metric = 0.325500)


In [12]:
# --- ДИАГНОСТИКА КАЛИБРОВКИ ---

bias_vals = np.array(list(biases.values()))
print(f"Биасы: min={bias_vals.min():,.0f}  max={bias_vals.max():,.0f}  "
      f"mean={bias_vals.mean():,.0f}  std={bias_vals.std():,.0f}")
print(f"Ненулевых биасов (|b|>1): {(np.abs(bias_vals) > 1).sum()} из {len(bias_vals)}")

# Среднее отклонение модели на валидации по каждой группе (= идеальный bias)
import pandas as pd
residuals_df = pd.DataFrame({
    "route_id": val_routes_arr,
    "step":     val_steps_arr,
    "residual": val_true_arr - val_preds_arr,   # true - pred > 0 → модель занижает
})
group_stats = residuals_df.groupby(["route_id", "step"])["residual"].agg(["mean", "median", "std"])
print("\nСтатистика остатков на валидации (по группам route×step):")
print(group_stats.describe().round(0))

# Метрика на валидации до/после
def compute_metric(y_true, y_pred):
    wape  = np.abs(y_pred - y_true).sum() / y_true.sum()
    rbias = abs(y_pred.sum() / y_true.sum() - 1)
    return wape + rbias, wape, rbias

val_bias_vec = np.array([biases.get((r, s), 0.0)
                          for r, s in zip(val_routes_arr, val_steps_arr)])
m_before = compute_metric(val_true_arr, val_preds_arr)
m_after  = compute_metric(val_true_arr, val_preds_arr + val_bias_vec)
print(f"\nМетрика на валидации ДО:  {m_before[0]:.6f}  (WAPE={m_before[1]:.4f}, RBias={m_before[2]:.4f})")
print(f"Метрика на валидации ПОСЛЕ: {m_after[0]:.6f}  (WAPE={m_after[1]:.4f}, RBias={m_after[2]:.4f})")


Биасы: min=-101,592  max=50,186  mean=-1,984  std=11,215
Ненулевых биасов (|b|>1): 8000 из 8000

Статистика остатков на валидации (по группам route×step):
          mean    median        std
count   8000.0    8000.0     8000.0
mean   -1984.0  -11980.0   113711.0
std     8490.0   12161.0    52707.0
min   -43171.0 -183115.0    11776.0
25%    -6016.0  -16930.0    87393.0
50%    -1343.0  -10396.0   109838.0
75%     2547.0   -5394.0   132199.0
max    81024.0   35533.0  1147558.0

Метрика на валидации ДО:  0.335353  (WAPE=0.3281, RBias=0.0073)
Метрика на валидации ПОСЛЕ: 0.326045  (WAPE=0.3260, RBias=0.0000)


## Сохранение результата

In [13]:
# --- SAVE SUBMISSION ---

submission_path = "submission.csv"
pred_df.to_csv(submission_path, index=False)
print("Saved:", submission_path)
print(pred_df.shape)

Saved: submission.csv
(8000, 2)
